# Eksploracja ekstrakcji struktury PDF — część 2

**Cel:** Dalsza ekstrakcja danych oparta o PyMuPDF w trybie `"dict"`.

## 1. Setup

In [ ]:
import pymupdf
from pathlib import Path
from collections import defaultdict

# PDF_PATH = Path("../data/raport_2024_wybrane_strony.pdf")
PDF_PATH = Path("../data/raport_2024_pl 1.pdf")
doc = pymupdf.open(PDF_PATH)
print(f"Liczba stron: {len(doc)}")

page0 = doc[0]
print(f"Wymiary strony 0: width={page0.rect.width:.1f}, height={page0.rect.height:.1f}")

## Sekcja 1 — Inspekcja surowej struktury "dict"

Załaduj kilka stron i wypisz bloki z: bbox, max rozmiar fontu, pierwsze 80 znaków tekstu.

In [ ]:
def inspect_page_blocks(page_idx: int):
    page = doc[page_idx]
    data = page.get_text("dict")
    print(f"\n{'='*60}")
    print(f"Strona {page_idx + 1} — {len(data['blocks'])} bloków")
    print(f"{'='*60}")
    for i, block in enumerate(data["blocks"]):
        if block["type"] != 0:  # pomiń bloki graficzne
            continue
        all_spans = [span for line in block["lines"] for span in line["spans"]]
        if not all_spans:
            continue
        max_size = max(s["size"] for s in all_spans)
        text = " ".join(s["text"] for s in all_spans).strip()
        bbox = block["bbox"]
        print(f"  [{i:2d}] bbox=({bbox[0]:.0f},{bbox[1]:.0f})-({bbox[2]:.0f},{bbox[3]:.0f})  "
              f"max_size={max_size:.1f}  tekst={repr(text[:80])}")

for idx in [9, 14, 15]:
    inspect_page_blocks(idx)

In [ ]:
# Czy są zakładki (TOC) — gotowa struktura sekcji!
toc = doc.get_toc()
if toc:
    print(f"\nZakładki ({len(toc)}):")
    for level, title, page in toc[:10]:
        print(f"  {'  ' * (level-1)}{title} → str. {page}")
else:
    print("Brak zakładek")

In [ ]:
# Jakie typy bloków dominują?
stats = {"text": 0, "image": 0}
for page in doc:
    for b in page.get_text("blocks"):
        stats["image" if b[6]==1 else "text"] += 1
print(f"Bloki: {stats}")

## Sekcja 3 — Rozmiary fontów i detekcja nagłówków

Zbierz wszystkie rozmiary fontów, kolor oraz typ fontu

In [ ]:
# ── zbieranie danych ──────────────────────────────────────────────
font_info = defaultdict(int)
examples  = defaultdict(list)   # max 3 przykłady na klucz

for page in doc:
    for b in page.get_text("dict")["blocks"]:
        for l in b.get("lines", []):
            for s in l["spans"]:
                sz    = round(s["size"])
                color = s["color"]
                font  = s["font"]
                text  = s["text"].strip()
                key   = (sz, color, font)

                font_info[key] += 1

                if text and len(examples[key]) < 3:
                    examples[key].append(text)

# ── helper: int → #RRGGBB ─────────────────────────────────────────
def int_to_hex(c: int) -> str:
    return f"#{c:06X}"

# ── sortuj malejąco po rozmiarze, potem po liczbie wystąpień ──────
sorted_info = sorted(font_info.items(), key=lambda x: (-x[0][0], -x[1]))

# ── wyświetl tabelę ───────────────────────────────────────────────
print(f"{'SIZE':>5}  {'COLOR':<9}  {'COUNT':>6}  FONT")
print("─" * 60)
for (sz, color, font), cnt in sorted_info:
    hex_color = int_to_hex(color)
    print(f"{sz:>5}  {hex_color:<9}  {cnt:>6}  {font}")

    for ex in examples[(sz, color, font)]:
        preview = ex[:80].replace("\n", " ")
        print(f"         └ '{preview}'")

In [ ]:
# ── sortuj malejąco po rozmiarze, potem po liczbie wystąpień ──────
sorted_info = sorted(font_info.items(), key=lambda x: (-x[1]))

# ── wyświetl tabelę ───────────────────────────────────────────────
print(f"{'SIZE':>5}  {'COLOR':<9}  {'COUNT':>6}  FONT")
print("─" * 60)
for (sz, color, font), cnt in sorted_info:
    hex_color = int_to_hex(color)
    print(f"{sz:>5}  {hex_color:<9}  {cnt:>6}  {font}")

    for ex in examples[(sz, color, font)]:
        preview = ex[:80].replace("\n", " ")
        print(f"         └ '{preview}'")

## Sekcja 5 — Wykrywanie hierarchi w dokumencie

### Ręczne określenie hierarchi

Tytuł: 33  #FFFFFF         3  TideSans-600Bunny
H1: 46  #FFFFFF        13  TideSans-600Bunny
H2: 42  #8B7350        92  TideSans-600Bunny
H3: 22  #8B7350       202  TideSans-600Bunny
H4: 19  #8B7350       192  TideSans-600Bunny
Text:18  #4B5559      6137  TideSansCond-300LilKahun
page_num 19  #FFFFFF       182  TideSansCond-400LilDude

Do wyrzucenia:
- wrzystkie fonty z rodziny Webdings

In [ ]:
from dataclasses import dataclass, field

@dataclass
class ExtractedPage:
    page_num: str | None = None
    title:  list[str] = field(default_factory=list)
    h1: list[str] = field(default_factory=list)
    h2: list[str] = field(default_factory=list)
    h3: list[str] = field(default_factory=list)
    h4: list[str] = field(default_factory=list)
    remaining: list[str] = field(default_factory=list)


STYLE_MAP = {
    ("TideSans-600Bunny", 33, "#FFFFFF"): "title",
    ("TideSans-600Bunny", 46, "#FFFFFF"): "h1",
    ("TideSans-600Bunny", 42, "#8B7350"): "h2",
    ("TideSans-600Bunny", 22, "#8B7350"): "h3",
    ("TideSans-600Bunny", 19, "#8B7350"): "h4",
    ("TideSansCond-400LilDude", 19, "#FFFFFF"): "page_num",
}

GARBAGE_FONTS = {"Webdings", "Wingdings-Regular"}


def is_garbage_font(font_name: str) -> bool:
    return any(font_name.startswith(g) for g in GARBAGE_FONTS)


def rgb_int_to_hex(color_int: int) -> str:
    return f"#{color_int:06X}"


def classify_block(font_name: str, font_size: int, color: str) -> str | None:
    if is_garbage_font(font_name):
        return None
    return STYLE_MAP.get((font_name, font_size, color), "remaining")


def extract_page(page: pymupdf.Page) -> ExtractedPage:
    extracted = ExtractedPage()

    for block in page.get_text("dict")["blocks"]:
        if block["type"] != 0:
            continue
        for line in block["lines"]:
            for span in line["spans"]:
                text = span["text"].strip()
                if not text:
                    continue

                hex_color = rgb_int_to_hex(span["color"])
                font_size = round(span["size"])
                category = classify_block(
                    font_name=span["font"],
                    font_size=font_size,
                    color=hex_color,
                )

                if category is None:
                    continue
                elif category == "title":
                    extracted.title.append(text)
                elif category == "h1":
                    extracted.h1.append(text)
                elif category == "h2":
                    extracted.h2.append(text)
                elif category == "h3":
                    extracted.h3.append(text)
                elif category == "h4":
                    extracted.h4.append(text)
                elif category == "page_num":
                    extracted.page_num = text
                else:
                    extracted.remaining.append({
                        "text": text,
                        "font": span["font"],
                        "size": font_size,
                        "color": hex_color,
                        "bbox": span["bbox"]
                    })

    return extracted


extracted_pages: list[ExtractedPage] = [extract_page(page) for page in doc]

In [ ]:
for page in extracted_pages[3:]:
    if page.remaining:
        print(f"--- Strona {page.page_num} ---")
        for item in page.remaining:
            print(item)

In [ ]:
def print_page_structure(pages: list[ExtractedPage]) -> None:
    for p in pages:
        if not any((p.title, p.h1, p.h2, p.h3)):
            continue
        label = f"str. {p.page_num}" if p.page_num else "str. ?"
        print(f"\n{'─'*55}")
        print(f"  {label}")
        print(f"{'─'*55}")
        for t in p.title: print(f"  TITLE : {t}")
        for h in p.h1:    print(f"  H1    : {h}")
        for h in p.h2:    print(f"  H2    : {h}")
        for h in p.h3:    print(f"  H3    : {h}")

print_page_structure(extracted_pages)

In [ ]:
import re

page2 = doc[1]
data  = page2.get_text("dict")

PATTERN_CHAPTER_NUM  = re.compile(r'^[IVXLCDM]+\s*$')
PATTERN_CHAPTER_FULL = re.compile(r'^([IVXLCDM]+)\s+(.+?)\s*$')
PATTERN_PAGE_ONLY    = re.compile(r'^\d+\s*$')
PATTERN_SUB_FULL     = re.compile(r'^\d+\.\s+(.+?)\s*\.{2,}\s*(\d+)\s*$')
PATTERN_SUB_START    = re.compile(r'^\d+\.\s+(.+)$')
RE_DOTS_END          = re.compile(r'^(.*?)\s*\.{2,}\s*(\d+)\s*$')

toc_entries = []
print("=== parsowanie strony 2 ===\n")

for block in data["blocks"]:
    if block["type"] != 0:
        continue
    lines = [
        "".join(s["text"] for s in line["spans"]).strip()
        for line in block["lines"]
    ]
    lines = [l for l in lines if l]

    i = 0
    while i < len(lines):
        line = lines[i]

        # 1. Nagłówek 3-liniowy: "I" / "Jedyny taki bank" / "3"
        if (PATTERN_CHAPTER_NUM.match(line)
                and i + 2 < len(lines)
                and PATTERN_PAGE_ONLY.match(lines[i + 2])):
            roman, title, pg = line.strip(), lines[i+1].strip(), lines[i+2].strip()
            toc_entries.append((f"{roman} {title}", pg))
            print(f"  CHAPTER3 str.{pg:>4}  {repr(roman + ' ' + title)}")
            i += 3
            continue

        # 2. Nagłówek 2-liniowy: "VIII Perspektywy" / "142"
        mc = PATTERN_CHAPTER_FULL.match(line)
        if (mc and i + 1 < len(lines)
                and PATTERN_PAGE_ONLY.match(lines[i + 1])):
            roman, title, pg = mc.group(1), mc.group(2).strip(), lines[i+1].strip()
            toc_entries.append((f"{roman} {title}", pg))
            print(f"  CHAPTER2 str.{pg:>4}  {repr(roman + ' ' + title)}")
            i += 2
            continue

        # 3. Podpunkt pełny na 1 linii: "N. tytuł ..... N"
        ms = PATTERN_SUB_FULL.match(line)
        if ms:
            title, pg = ms.group(1).strip(), ms.group(2)
            toc_entries.append((title, pg))
            print(f"  SUB      str.{pg:>4}  {repr(line)}")
            i += 1
            continue

        # 4. Podpunkt zawijany: start na linii i, kontynuacja+dots na linii i+1
        ms2 = PATTERN_SUB_START.match(line)
        if ms2 and i + 1 < len(lines):
            m2 = RE_DOTS_END.match(lines[i + 1])
            if m2:
                prefix = ms2.group(1).strip()
                cont   = m2.group(1).strip()
                pg     = m2.group(2)
                title  = f"{prefix} {cont}".strip()
                toc_entries.append((title, pg))
                print(f"  SUB-WRAP str.{pg:>4}  {repr(line)}")
                print(f"           +cont  {repr(lines[i+1])}")
                i += 2
                continue

        print(f"  skip             {repr(line)}")
        i += 1

print(f"\nZnaleziono {len(toc_entries)} wpisów TOC")

In [ ]:
# Komórka C — porównanie TOC vs ekstrakcja
extracted_by_page: dict[str, list[str]] = {}
for p in extracted_pages:
    if p.page_num is None:
        continue
    extracted_by_page[p.page_num] = p.title + p.h1 + p.h2

print(f"\n{'TOC ENTRY':<50} {'STR':>5}  MATCH")
print("─" * 70)
for toc_title, toc_page in toc_entries:
    found = extracted_by_page.get(toc_page, [])
    toc_lower = toc_title.lower()
    hit = any(toc_lower in h.lower() or h.lower() in toc_lower for h in found)
    mark = "OK" if hit else "BRAK"
    print(f"{toc_title[:50]:<50} {toc_page:>5}  {mark}")

## Wnioski

### Detekcja hierarchii przez fonty — działa

Hierarchię dokumentu (tytuł, H1–H2) można wydobyć wyłącznie na podstawie
informacji o fontach: nazwa kroju, rozmiar i kolor. Nie są potrzebne zakładki PDF
ani inne metadane strukturalne.

Podejście zostało zweryfikowane ze spisem treści ze strony 2.

### Kolejne kroki ekstrakcji

1. **Wydobywanie treści podrozdziałów** — przypisanie akapitów tekstu do
   właściwego nagłówka nadrzędnego (H2).

2. **Usunięcie nawigacji bocznej** — bloki z lewej kolumny (x1 < ~154) to menu
   nawigacyjne, które nie niesie treści merytorycznej dla LLM.

3. **Filtrowanie treści nieinterpretowalnych przez LLM** — fonty ozdobne
   (Webdings, Wingdings) zawierają symbole bez znaczenia semantycznego
   i powinny być odfiltrowane.

4. **Wykrywanie tabel i ich obróbka pod LLM** — tabele w PDF wymagają osobnego podejścia i konwersji do formatu tekstowego czytelnego dla modelu językowego.

### Plan na part3

Na potrzeby `explore_extraction_part3` raport zostanie podzielony na mniejsze
pliki PDF (rodziały). Na tych fragmentach będzie dopracowywany
sposób poprawnej ekstrakcji treści z poszczególnych typów stron.

In [ ]:
import re
from pathlib import Path

chapters_dir = Path("../data/chapters")
chapters_dir.mkdir(exist_ok=True)

def sanitize_filename(text: str) -> str:
    text = re.sub(r'[<>:"/\\|?*]', '', text)
    text = re.sub(r'\s+', '_', text.strip())
    return text[:60]

# Strony z H1 i widocznym numerem strony = granica rozdziału
# (strony promo/okładki mają H1 ale page_num=None — pomijamy je)
chapter_starts: list[tuple[int, str]] = []
for i, page in enumerate(extracted_pages):
    if page.h1 and page.page_num is not None:
        chapter_starts.append((i, page.h1[0]))

print(f"Znaleziono {len(chapter_starts)} rozdziałów:\n")
for idx, name in chapter_starts:
    pg = extracted_pages[idx].page_num
    end_idx = next((s for s, _ in chapter_starts if s > idx), len(doc))
    print(f"  [{idx:2d}] str.wyś={str(pg):>4}  pages={end_idx - idx}  {name}")

In [ ]:
saved = []
for i, (start_idx, chapter_name) in enumerate(chapter_starts):
    is_last = (i + 1 == len(chapter_starts))
    if is_last:
        end_idx = len(doc) - 1      # pomija ostatnią stronę dokumentu
    else:
        end_idx = chapter_starts[i + 1][0]

    filename = f"{i+1:02d}_{sanitize_filename(chapter_name)}.pdf"
    output_path = chapters_dir / filename

    chapter_doc = pymupdf.open()
    chapter_doc.insert_pdf(doc, from_page=start_idx, to_page=end_idx - 1)
    chapter_doc.save(str(output_path))
    chapter_doc.close()

    pages_count = end_idx - start_idx
    saved.append(filename)
    print(f"  Zapisano: {filename}  ({pages_count} str.)")

print(f"\nRazem {len(saved)} plików → {chapters_dir.resolve()}")